# Exp17 — Left-Relation MLP Ablation (Qwen2-VL-2B)
Tests whether layers 21 & 23 are causal for **left** spatial reasoning too.
If yes → circuit is relation-general, not a vflip artifact.

In [ ]:
# 1. Install dependencies
!pip install transformers accelerate qwen-vl-utils -q

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# 2. Clone repo
import os
if not os.path.exists('Mechanistic-interpretability-research'):
    !git clone https://github.com/likithp82/Mechanistic-interpretability-research.git
%cd Mechanistic-interpretability-research
!git log --oneline -3

In [ ]:
# 3. Download COCO val2017 images and reconstruct manifest images
# The manifest stores custom filenames (e.g. 000000143068__143068__430155_623899__original.jpg)
# that are NOT the raw COCO filenames — they must be built from the source COCO image + transform.
import os, json, shutil
from PIL import Image

os.makedirs('src/dataset/spatial_real_dataset_val2017/images', exist_ok=True)

if not os.path.exists('/tmp/val2017.zip'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/val2017.zip
    !unzip -q /tmp/val2017.zip -d /tmp/

manifest_rows = []
with open('src/dataset/spatial_real_dataset_val2017/manifest.jsonl') as f:
    for line in f:
        manifest_rows.append(json.loads(line))

# Only build images needed for --relation left (original transform, left, label=1)
target_rows = [
    r for r in manifest_rows
    if r.get('transform') == 'original'
    and r.get('true_relation') == 'left'
    and r.get('label') == 1
]
print(f'Target rows for left/original/label=1: {len(target_rows)}')

dest = 'src/dataset/spatial_real_dataset_val2017/images'
built = 0
skipped = 0
for row in target_rows:
    out_name = row['output_file'].split('/')[-1]
    dst = f'{dest}/{out_name}'
    if os.path.exists(dst):
        built += 1
        continue
    src = f'/tmp/val2017/{row["source_file"]}'
    if not os.path.exists(src):
        skipped += 1
        continue
    # original transform = direct copy (no pixel changes)
    shutil.copy2(src, dst)
    built += 1

print(f'Built {built}/{len(target_rows)} images (source not found in COCO: {skipped})')
if skipped > 0:
    print('WARNING: some source images missing — model-correct count may be lower than max-images')


In [ ]:
# 4. Run Exp17 — MLP-only sweep for 'left' relation, layers 18-24
# ~17 min on T4 for MLP phase (7 layers × 60 images)
!PYTHONUNBUFFERED=1 python -u src/day7/exp15_attention_head_sweep.py \
    --manifest src/dataset/spatial_real_dataset_val2017/manifest.jsonl \
    --images-root src/dataset/spatial_real_dataset_val2017/images \
    --max-images 60 \
    --layer-window 3 \
    --ablate-mlp \
    --skip-heads \
    --relation left \
    --out-dir src/day7 \
    2>&1 | tee src/day7/exp17_left_mlp_run.log


In [ ]:
# 5. Print results summary
import json
with open('src/day7/results_exp15.json') as f:
    r = json.load(f)

print(f"Relation: {r.get('relation', 'above')}")
print(f"Question: {r['fixed_question']}")
print()
print('MLP flip rates by layer:')
for layer, fr in sorted(r['mlp_flip_rates'].items(), key=lambda x: int(x[0])):
    bar = '█' * int(float(fr) * 30)
    print(f'  Layer {int(layer):2d}: {float(fr):.4f}  {bar}')